# Decision Transformer en Hopper

Notebook opcional del **módulo 17** del curso (en la app: `uv run dev` y la ruta `#/modulo/rl-decision-transformer`). Usa modelos reales; el módulo interactivo explica la teoría con NumPy.

> **Requisitos:** desde la raíz del repositorio, `uv sync --group notebooks` y abre este notebook con `uv run --group notebooks jupyter lab`. La primera ejecución descarga los modelos de Hugging Face.

Necesita `gymnasium[mujoco]` (`uv pip install "gymnasium[mujoco]"`).

In [ ]:
import gymnasium as gym
import numpy as np
import torch
from transformers import DecisionTransformerModel

modelo = DecisionTransformerModel.from_pretrained("edbeeching/decision-transformer-gym-hopper-expert")
modelo.eval()
env = gym.make("Hopper-v4")
d_s, d_a = env.observation_space.shape[0], env.action_space.shape[0]
# Media y desviación de los estados del dataset con el que se entrenó (de la tarjeta del modelo)
media = np.array([1.3490015, -0.11208222, -0.5506444, -0.13188992, -0.00378754, 2.6071432, 0.02322114, -0.01626922, -0.06840388, -0.05183131, 0.04272673])
desv = np.array([0.15980862, 0.0446214, 0.14307782, 0.17629202, 0.5912333, 0.5899924, 1.5405099, 0.8152689, 2.0173461, 2.4107876, 5.8440027])

In [ ]:
def episodio(retorno_objetivo, escala=1000.0, K=20, max_pasos=1000):
    s, _ = env.reset(seed=0)
    S = torch.zeros(1, 0, d_s)
    A = torch.zeros(1, 0, d_a)
    R = torch.zeros(1, 0, 1)
    T = torch.zeros(1, 0, dtype=torch.long)
    rtg, total = retorno_objetivo / escala, 0.0
    for t in range(max_pasos):
        S = torch.cat([S, torch.tensor((s - media) / desv, dtype=torch.float32)[None, None]], 1)[:, -K:]
        A = torch.cat([A, torch.zeros(1, 1, d_a)], 1)[:, -K:]
        R = torch.cat([R, torch.tensor([[[rtg]]], dtype=torch.float32)], 1)[:, -K:]
        T = torch.cat([T, torch.tensor([[t]])], 1)[:, -K:]
        with torch.no_grad():
            _, a, _ = modelo(states=S, actions=A, returns_to_go=R, timesteps=T, attention_mask=torch.ones_like(T), return_dict=False)
        accion = a[0, -1].numpy()
        A[0, -1] = a[0, -1]
        s, r, fin, trunc, _ = env.step(accion)
        total += r
        rtg -= r / escala
        if fin or trunc:
            break
    return total

for objetivo in [1000, 2000, 3600]:
    print(objetivo, "->", round(episodio(objetivo), 1))